# **Implementación del Algoritmo de Shor para resolver el  DLP**

## Oráculo de Montgomery

### Rodrigo Hernández Sacristán

#### Máster en Computación Cuántica, Universidad Internacional de la Rioja (UNIR)

In [2]:
## LIBRERÍAS NECESARIAS 

import numpy as np
import math
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate 
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit.quantum_info import Operator
from qiskit.circuit.library import QFT
import warnings
import random

In [ ]:
"""
Oráculo cuántico para Shor-DLP con MULTIPLICACIÓN MODULAR DE MONTGOMERY

    - Exponenciación modular en  O(n^3)   
    - Número de ancillas: 2n+1     

    |a>|b>||0..0>  ->  |a>|b>| g^a * h^(-b) mod p >

Recuento de qubits (para módulo p, con n = ceil(log2 p)):
    reg_C (valor) = n           <- tu registro de valor (smallreg)
    reg_anc       = 2n + 1       <- espacio de trabajo Montgomery
Por ejemplo para p=23 (n=5): reg_anc = 11 qubits, total del oráculo = 3n + (2n+1) = 26.

La clave de Montgomery: la reducción modular extrae el bit MENOS significativo
con una sola Hadamard (en vez del bit MÁS significativo con una QFT completa
en cada suma, como en Beauregard). Eso es lo que baja el coste a O(n^3).

"""

warnings.filterwarnings("ignore", category=DeprecationWarning)


def _qft(m):
    return QFT(m, do_swaps=False).to_gate()


def _iqft(m):
    return QFT(m, do_swaps=False).inverse().to_gate()


def _angle(k, b):
    """Ángulo de Draper: b*pi / 2^k."""
    return b * np.pi / (2 ** k)


# ------------------------------------------------------------------
# Sumadores de Draper en espacio de Fourier
# ------------------------------------------------------------------
def _ccp(theta):
    """Puerta de fase doble-controlada mediante fases simples (medio ángulo)."""
    ctrl = QuantumRegister(2, "ctrl")
    q = QuantumRegister(1, "q")
    c = QuantumCircuit(ctrl, q, name="CCP")
    c.cp(theta / 2, ctrl[1], q)
    c.cx(ctrl[0], ctrl[1])
    c.cp(-theta / 2, ctrl[1], q)
    c.cx(ctrl[0], ctrl[1])
    c.cp(theta / 2, ctrl[0], q)
    return c.to_gate()


def _phi_add_cc(n, b, factor):
    """Suma doble-controlada: |phi(x)> -> |phi(x + factor*b)>."""
    ctrl = QuantumRegister(2, "c")
    reg = QuantumRegister(n, "r")
    a = QuantumCircuit(ctrl, reg, name="CCADD")
    for k in range(n):
        a.append(_ccp(factor * _angle(k, b)), list(ctrl) + [reg[k]])
    return a.to_gate()


def _phi_add_c_ignore(n, b, factor, ignorebits):
    """Suma controlada que IGNORA los 'ignorebits' bits menos significativos
    (los ya fijados en |u> durante la reducción de Montgomery)."""
    ctrl = QuantumRegister(1, "c")
    reg = QuantumRegister(n - ignorebits, "r")
    a = QuantumCircuit(ctrl, reg, name="CADDign")
    for k in range(n):
        if k < ignorebits:
            continue
        a.cp(factor * _angle(k, b), ctrl[0], reg[k - ignorebits])
    return a.to_gate()


def _phi_add_c(n, b, factor):
    return _phi_add_c_ignore(n, b, factor, 0)


# ------------------------------------------------------------------
# Multiplicación modular de Montgomery
# ------------------------------------------------------------------
def _mult_montgomery_partial_c(n, y_montg, p):
    """Multiplicación fuera de sitio + reducción de Montgomery:
    |x>|0..0> -> |x>|0>|xy mod p>|0..0>.  'y_montg' entra en forma de Montgomery (yR mod p)."""
    ctrl = QuantumRegister(1, "ctrl")
    small = QuantumRegister(n, "small")
    big = QuantumRegister(2 * n + 1, "big")
    m = QuantumCircuit(ctrl, small, big, name="MMULp")

    # (1) multiplicación por sumas repetidas en espacio de Fourier
    m.append(_qft(2 * n + 1), big)
    for i in range(n):
        s = (2 ** i * y_montg) % p
        m.append(_phi_add_cc(2 * n + 1, s, 1), [ctrl[0], small[i]] + list(big))

    # (2) reducción de Montgomery: en cada iteración se extrae el LSB con una
    #     Hadamard y se usa para controlar una resta de p; la división por 2 es
    #     implícita porque se trabaja sobre un registro cada vez más pequeño.
    for i in range(n):
        qubits = [big[i]] + list(big[i + 1:])
        m.h(big[i])
        m.append(_phi_add_c_ignore(2 * n + 1 - i, int(p), -1, 1), qubits)

    # (3) corrección de signo: extraer el bit de signo y sumar p si es negativo
    m.append(_iqft(n + 1), big[n:])
    signbit = big[-1]
    m.append(_qft(n), big[n:2 * n])
    m.append(_phi_add_c(n, p, 1), [signbit] + list(big[n:2 * n]))
    m.h(big[n]); m.cx(big[n], signbit); m.h(big[n])

    # (4) descomputación del registro u (restas con p^{-1} mod 2^{n+1})
    uncom = list(big[:n]) + [big[-1]]
    m.append(_qft(n + 1), uncom)
    pinv = pow(p, -1, 2 ** (n + 1))
    for i in range(n):
        s = ((2 ** i * y_montg % p) * pinv) % 2 ** (n + 1)
        m.append(_phi_add_cc(n + 1, s, -1), [ctrl[0], small[i]] + uncom)

    m.append(_iqft(n), big[n:2 * n])
    m.append(_iqft(n + 1), uncom)
    return m.to_gate()


def _mult_montgomery_c(n, y, p):
    """Multiplicación modular controlada EN SITIO: |c>|x>|0> -> |c>| x*y mod p >|0>.
    Dos multiplicaciones fuera de sitio (con y y con y^{-1}) más un SWAP."""
    ctrl = QuantumRegister(1, "ctrl")
    small = QuantumRegister(n, "small")
    big = QuantumRegister(2 * n + 1, "big")
    m = QuantumCircuit(ctrl, small, big, name="MMUL")

    R = 2 ** n
    y_montg = y * R % p
    m.append(_mult_montgomery_partial_c(n, y_montg, p), m.qubits)
    for i in range(n):
        m.cswap(ctrl[0], small[i], big[n + i])
    iy_montg = pow(y, -1, p) * R % p
    m.append(_mult_montgomery_partial_c(n, iy_montg, p).inverse(), m.qubits)
    return m.to_gate()


def _mod_exp_montgomery(n, a, p):
    """|x>|input> -> |x>| valor*a^x mod p >.  input = 3n+1 qubits: [valor(n) | big(2n+1)]."""
    top = QuantumRegister(n, "top")
    bot = QuantumRegister(3 * n + 1, "bot")
    c = QuantumCircuit(top, bot, name=f"{a}^x mod {p}")
    for i in range(n):
        cte = pow(a, 2 ** i, p)
        if cte == 1:          # multiplicar por 1 es la identidad -> se salta
            continue
        c.append(_mult_montgomery_c(n, cte, p), [top[i]] + list(bot))
    return c.to_gate()



def num_ancillas_montgomery(p):
    """Ancillas que necesita el oráculo Montgomery: 2n+1 (para p=23 -> 11)."""
    return 2 * math.ceil(math.log2(p)) + 1


def aplicar_oraculo_montgomery(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc):
    """
    Implementa f(a,b) = g^a * h^(-b) mod p con multiplicación de Montgomery.
    """
    n = len(reg_A)
    size = math.ceil(math.log2(p))
    assert len(reg_C) >= size, f"reg_C necesita >= {size} qubits para p={p}"
    assert len(reg_anc) == num_ancillas_montgomery(p), \
        f"reg_anc debe tener {num_ancillas_montgomery(p)} qubits para p={p}, tiene {len(reg_anc)}"

    bottom = list(reg_C) + list(reg_anc)   # smallreg(n) + big(2n+1)
    qc.x(reg_C[0])                         # valor inicial |1>
    h_inv = pow(h, -1, p)
    qc.append(_mod_exp_montgomery(n, g, p),     list(reg_A) + bottom)
    qc.append(_mod_exp_montgomery(n, h_inv, p), list(reg_B) + bottom)